In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.utils import to_categorical

c:\Users\harip\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [ ]:

df = pd.read_csv(r"D:\Sentimental analysis\data\cleaned_chatgpt_reviews_updated.csv")

In [9]:
# Safety: Drop empty rows
df.dropna(subset=['cleaned_review'], inplace=True)

In [10]:
# Split FIRST (Keep Test data pure/unbalanced to reflect real world)
X_train_raw, X_test_text, y_train_raw, y_test = train_test_split(
    df['cleaned_review'], 
    df['sentiment_encoded'],  # Use existing encoded column (0=Negative, 1=Neutral, 2=Positive)
    test_size=0.2, 
    random_state=42
)

In [11]:
# Combine temporary training data for balancing
train_data = pd.DataFrame({'text': X_train_raw, 'label': y_train_raw})

In [12]:
# Separate classes (Assuming 0=Neg, 1=Neu, 2=Pos)
# If  label_encoded is different, change these numbers!
neg_df = train_data[train_data['label'] == 0]
neu_df = train_data[train_data['label'] == 1]
pos_df = train_data[train_data['label'] == 2]

In [13]:
# Find the biggest class size to match
max_size = max(len(neg_df), len(neu_df), len(pos_df))

In [14]:
# Upsample (Clone) everyone to match the biggest size
neg_bal = resample(neg_df, replace=True, n_samples=max_size, random_state=42)
neu_bal = resample(neu_df, replace=True, n_samples=max_size, random_state=42)
pos_bal = resample(pos_df, replace=True, n_samples=max_size, random_state=42)

In [15]:
# Combine back together
train_balanced = pd.concat([neg_bal, neu_bal, pos_bal])
X_train_final = train_balanced['text']
y_train_final = train_balanced['label']

print(f"✅ Data Balanced! New Training Size: {len(X_train_final)}")

✅ Data Balanced! New Training Size: 483


## TOKENIZATION (Prepare for LSTM)

In [16]:
print("🔢 Step 3: Converting Text to Numbers...")

vocab_size = 5000
max_length = 100

🔢 Step 3: Converting Text to Numbers...


In [17]:
# Train Tokenizer 
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train_final)

In [18]:
# Convert to sequences
train_sequences = tokenizer.texts_to_sequences(X_train_final)
test_sequences = tokenizer.texts_to_sequences(X_test_text)

In [19]:
# Convert to sequences
train_sequences = tokenizer.texts_to_sequences(X_train_final)
test_sequences = tokenizer.texts_to_sequences(X_test_text)

In [20]:
# Pad sequences
X_train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post')
X_test_padded = pad_sequences(test_sequences, maxlen=max_length, padding='post')

In [21]:
# One-Hot Encode labels
y_train_cat = to_categorical(y_train_final, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

## TRAIN LSTM

In [22]:
from tensorflow.keras.regularizers import l2
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_length),
    Bidirectional(LSTM(64)),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

c:\Users\harip\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [23]:
# Train
history = model.fit(X_train_padded, y_train_cat, 
                    epochs=5, 
                    validation_data=(X_test_padded, y_test_cat),
                    verbose=1)

Epoch 1/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - accuracy: 0.3872 - loss: 1.0626 - val_accuracy: 0.9200 - val_loss: 0.9617
Epoch 2/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9048 - loss: 0.7733 - val_accuracy: 1.0000 - val_loss: 0.4246
Epoch 3/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9876 - loss: 0.2267 - val_accuracy: 1.0000 - val_loss: 0.0452
Epoch 4/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9959 - loss: 0.0357 - val_accuracy: 1.0000 - val_loss: 0.0073
Epoch 5/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 1.0000 - loss: 0.0080 - val_accuracy: 1.0000 - val_loss: 0.0023


## Evaluvate

In [24]:
# Evaluate
y_pred_probs = model.predict(X_test_padded)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

print("\n--- 🏆 FINAL REPORT CARD ---")
print(classification_report(y_test, y_pred_classes, target_names=['Negative', 'Neutral', 'Positive']))

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step

--- 🏆 FINAL REPORT CARD ---
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        36
     Neutral       1.00      1.00      1.00        25
    Positive       1.00      1.00      1.00        39

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [35]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# Download resources (just in case)
nltk.download('stopwords')
nltk.download('wordnet')

# 1. DEFINE SAFE CLEANING (Keeps 'not', 'no', 'bad')
def safe_clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    stop_words = set(stopwords.words('english'))
    
    # 🛡️ PROTECT NEGATIVE WORDS
    for w in ['not', 'no', 'nor', 'but', "don't", "won't", "didn't", "couldn't"]:
        if w in stop_words:
            stop_words.remove(w)
            
    # Remove only useless words
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\harip\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [36]:
print("🧹 Applying safe cleaning...")
if 'review' in df.columns:
    df['cleaned_review'] = df['review'].apply(safe_clean_text)
else:
    print("⚠️ Warning: 'review' column not found. Using existing 'cleaned_review'.")

print("✅ Data is fresh and includes negative words!")

🧹 Applying safe cleaning...
✅ Data is fresh and includes negative words!


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("Preparing Data for ML Models...")

# 1. CONVERT TO TF-IDF (Word Counts)

tfidf = TfidfVectorizer(max_features=5000)

# Fit on the Balanced Training Data
X_train_tfidf = tfidf.fit_transform(X_train_final)

# Transform the Test Data (Do NOT fit on test data!)
X_test_tfidf = tfidf.transform(X_test_text)

print("Data Converted to TF-IDF!")

# 2. DEFINE THE MODELS
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000,class_weight='balanced'),
    "Naïve Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42) 
    # Note: We don't need class_weight='balanced' here because WE already balanced the data!
}

# 3. TRAIN AND EVALUATE LOOP
for name, model in models.items():
    print(f"\n Training {name}...")
    
    # Train on the Balanced Data
    model.fit(X_train_tfidf, y_train_final)
    
    # Predict
    y_pred = model.predict(X_test_tfidf)
    
    # Report
    print(f"--- {name} Results ---")
    # 0=Neg, 1=Neu, 2=Pos
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

Preparing Data for ML Models...
Data Converted to TF-IDF!

 Training Logistic Regression...
--- Logistic Regression Results ---
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        36
     Neutral       1.00      1.00      1.00        25
    Positive       1.00      1.00      1.00        39

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100


 Training Naïve Bayes...
--- Naïve Bayes Results ---
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        36
     Neutral       1.00      1.00      1.00        25
    Positive       1.00      1.00      1.00        39

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100


 Training Random Forest...
--- Random Forest Results --

In [26]:
print(f"Train Shape: {X_train_tfidf.shape}")
print(f"Test Shape:  {X_test_tfidf.shape}")

Train Shape: (483, 76)
Test Shape:  (100, 76)


In [38]:
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
print("TF-IDF Vectorizer saved as 'tfidf_vectorizer.pkl'")
with open("logistic_regression_model.pkl", "wb") as f:
    pickle.dump(models["Logistic Regression"], f)

TF-IDF Vectorizer saved as 'tfidf_vectorizer.pkl'
